# Arkhe ARC3 observation explorer v1

Experimental observation-only frontier policy, exact 160-action cap. Native ContainmentGate + unsigned WER + in-memory WormGraph readback. Development policy only. Local comparison did not demonstrate an advantage. No LLM, game-specific rules, hidden engine access, production signatures or AGI claim.

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /tmp/my_agent.py
"""Observation-only frontier explorer. Native development gate is optional/explicit.

No game-specific IDs, hidden engine access, model provider or pre-solved routes.
This is a first experimental policy, not a demonstrated general solver.
"""
from __future__ import annotations
from collections import Counter, deque
import hashlib
import json
import os
from pathlib import Path
import random
import subprocess
from arcengine import GameAction, GameState
from agents.agent import Agent


def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)


def snapshot(frame):
    return {"frame":frame.frame,"state":frame.state.name,
            "levels_completed":frame.levels_completed,
            "available_actions":list(frame.available_actions)}


def state_key(frame):
    # guid/action_input are transport history, not perceptual state.
    return hashlib.sha256(canonical(snapshot(frame)).encode()).hexdigest()


def candidates(frame):
    legal = sorted({int(a) for a in frame.available_actions if 1 <= int(a) <= 7})
    result = [(a, None, None) for a in legal if a != 6]
    if 6 in legal:
        # Palette connected components identify click targets from pixels only.
        grid = frame.frame[-1] if frame.frame else []
        height=len(grid); width=len(grid[0]) if height else 0
        seen=set(); regions=[]
        for y in range(height):
            for x in range(width):
                if (x,y) in seen: continue
                color=grid[y][x]; todo=[(x,y)]; seen.add((x,y)); cells=[]
                while todo:
                    cx,cy=todo.pop();cells.append((cx,cy))
                    for nx,ny in ((cx+1,cy),(cx-1,cy),(cx,cy+1),(cx,cy-1)):
                        if 0<=nx<width and 0<=ny<height and (nx,ny) not in seen and grid[ny][nx]==color:
                            seen.add((nx,ny));todo.append((nx,ny))
                if len(cells)<=max(1,width*height//3):
                    # Select an actual component pixel nearest its centroid.
                    mx=sum(c[0] for c in cells)/len(cells);my=sum(c[1] for c in cells)/len(cells)
                    center=min(cells,key=lambda c:((c[0]-mx)**2+(c[1]-my)**2,c))
                    regions.append((len(cells),center))
        points=[c for _,c in sorted(regions)[:48]]
        points += [(x,y) for y in (8,24,40,56) for x in (8,24,40,56)]
        result += [(6,x,y) for x,y in sorted(set(points)) if x<64 and y<64]
    return result


class Explorer:
    """Observed deterministic transition graph; contradictory edges are discarded."""
    def __init__(self, seed=0, mode="frontier"):
        self.rng=random.Random(seed);self.mode=mode
        self.tried=Counter();self.edges={};self.unstable=set();self.options={};self.visits=Counter()

    def observe(self, source, action, target):
        edge=(source,action); self.tried[edge]+=1;self.visits[target]+=1
        if edge in self.edges and self.edges[edge]!=target:
            self.unstable.add(edge);self.edges.pop(edge,None)
        elif edge not in self.unstable:
            self.edges[edge]=target

    def choose(self, key, options):
        if not options: raise RuntimeError("No legal actions advertised")
        self.options[key]=tuple(options)
        if self.mode=="random": return self.rng.choice(options)
        untried=[a for a in options if not self.tried[key,a]]
        if untried: return self.rng.choice(untried)
        # Route through observed transitions to a state with untried actions.
        queue=deque([(key,None)]);seen={key}
        while queue and len(seen)<=4096:
            node,first=queue.popleft()
            if node!=key and any(not self.tried[node,a] for a in self.options.get(node,())):
                return first
            for action in self.options.get(node,()):
                target=self.edges.get((node,action))
                if target and target not in seen:
                    seen.add(target);queue.append((target,action if first is None else first))
        ranks={a:(self.tried[key,a],self.visits[self.edges.get((key,a),key)]) for a in options}
        best=min(ranks.values());return self.rng.choice([a for a in options if ranks[a]==best])


class MyAgent(Agent):
    MAX_ACTIONS=160

    def __init__(self,*args,**kwargs):
        super().__init__(*args,**kwargs)
        self.action_counter=0;self.dispatched=0
        self.explorer=Explorer(int(os.getenv("ARKHE_ARC_SEED","0")),os.getenv("ARKHE_ARC_POLICY","frontier"))
        self.bridge=os.getenv("ARKHE_ARC_GATE_EXE")
        self.trace_path=os.getenv("ARKHE_ARC_TRACE")
        self.manifest={"profile":"arc-observation-explorer-v1","seed":os.getenv("ARKHE_ARC_SEED","0"),
                       "policy":self.explorer.mode,"max_actions":self.MAX_ACTIONS,
                       "agent_sha256":hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
                       "bridge_sha256":hashlib.sha256(Path(self.bridge).read_bytes()).hexdigest() if self.bridge else None}
        self.previous_hash="0"*64

    def is_done(self,frames,latest_frame):
        return latest_frame.state is GameState.WIN or self.dispatched>=self.MAX_ACTIONS

    def choose_action(self,frames,latest_frame):
        self.current=latest_frame
        if latest_frame.state in (GameState.NOT_PLAYED,GameState.GAME_OVER):
            self.selected=(0,None,None)
        else:
            self.selected=self.explorer.choose(state_key(latest_frame),candidates(latest_frame))
        a,x,y=self.selected;action=GameAction['RESET' if a==0 else f'ACTION{a}']
        if a==6: action.set_data({"x":x,"y":y})
        action.reasoning={"policy":self.explorer.mode,"native_gate":"ENABLED" if self.bridge else "NOT_INTEGRATED"}
        return action

    def native(self,payload):
        result=subprocess.run([self.bridge],input=canonical(payload),text=True,capture_output=True,timeout=15)
        if result.returncode: raise RuntimeError("Native ARC gate process failed: "+result.stderr[:400])
        value=json.loads(result.stdout)
        if value.get("gate")!="ALLOW": raise RuntimeError("Native ARC gate rejected action")
        return value

    def take_action(self,action):
        if self.dispatched>=self.MAX_ACTIONS: raise RuntimeError("Action budget exhausted before dispatch")
        a,x,y=self.selected
        payload={"run_id":f"{self.game_id}:{self.dispatched}:{hashlib.sha256(canonical(self.manifest).encode()).hexdigest()}",
                 "phase":"preflight","observation":snapshot(self.current),"action":{"id":a}}
        if a==6: payload["action"].update(x=x,y=y)
        pre=self.native(payload) if self.bridge else None
        self.dispatched+=1
        # A transport exception is fatal; its dispatch still consumes the budget.
        outcome=super().take_action(action)
        if outcome is None: raise RuntimeError("Environment returned no outcome")
        post=self.native({**payload,"phase":"record","outcome":snapshot(outcome)}) if self.bridge else None
        if pre and pre["intent_digest"]!=post["intent_digest"]:
            raise RuntimeError("Native intent changed between preflight and record")
        source=state_key(self.current);target=state_key(outcome)
        # Reset/death transitions are not planned as reversible navigation edges.
        if a and outcome.state is not GameState.GAME_OVER:
            self.explorer.observe(source,self.selected,target)
        else:
            self.explorer.tried[source,self.selected]+=1
        record={"game_id":self.game_id,"step":self.dispatched,"before":source,"after":target,
                "action":payload["action"],"state":outcome.state.name,"levels_completed":outcome.levels_completed,
                "native_preflight":pre,"native_record":post,"manifest":self.manifest,
                "previous_hash":self.previous_hash,"trace_trust":"UNSIGNED_HOST_HASH_CHAIN"}
        self.previous_hash=hashlib.sha256(canonical(record).encode()).hexdigest();record["record_hash"]=self.previous_hash
        if self.trace_path:
            with open(self.trace_path,"a",encoding="utf-8") as f:f.write(canonical(record)+"\n")
        return outcome


In [ ]:
import base64, gzip, hashlib, json, os, pathlib, subprocess
gate = pathlib.Path('/tmp/arkhe_arc_action_gate')
binary = gzip.decompress(base64.b64decode('...'))
assert hashlib.sha256(binary).hexdigest() == 'ccf176b7df4fbcf70da12ec1c6ec4b8b0e553ae0ede1230d1865e6b0292e75c8'
gate.write_bytes(binary)
gate.chmod(0o700)
request = {'run_id':'kaggle-native-selftest','phase':'preflight',
           'observation':{'frame':[[[0]]],'state':'NOT_FINISHED','levels_completed':0,'available_actions':[1]},
           'action':{'id':1}}
def native_call(payload):
    p = subprocess.run([str(gate)], input=json.dumps(payload), text=True, capture_output=True, timeout=15, check=True)
    return json.loads(p.stdout)
pre = native_call(request)
assert pre['gate'] == 'ALLOW'
post = native_call({**request,'phase':'record','outcome':request['observation']})
assert post['gate'] == 'ALLOW' and post['intent_digest'] == pre['intent_digest']
assert post['wormgraph'] == 'IN_MEMORY_READBACK_ONLY'
assert native_call({**request,'emergency_stop':True})['gate'] == 'REJECT'
os.environ['ARKHE_ARC_GATE_EXE'] = str(gate)
os.environ['ARKHE_ARC_SEED'] = '0'
os.environ['ARKHE_ARC_POLICY'] = 'frontier'
os.environ['ARKHE_ARC_TRACE'] = '/kaggle/working/arkhe-actions.jsonl'
pathlib.Path('/kaggle/working/native-selftest.json').write_text(json.dumps({
    'status':'PASS','binary_sha256':'ccf176b7df4fbcf70da12ec1c6ec4b8b0e553ae0ede1230d1865e6b0292e75c8','policy':'DEV_UNPROTECTED',
    'signature':'UNSIGNED','wormgraph':'IN_MEMORY_READBACK_ONLY',
    'intent_digest':pre['intent_digest'],'worker_digest':post['worker_digest']}))
print('Arkhe native bootstrap: PASS (unprotected simulation, unsigned evidence)')


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the gateway sidecar to be ready.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the framework into a writable location.
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Drop our agent in as a framework template.
    !cp /tmp/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Register MyAgent in the framework's agent registry. We rewrite
    # __init__.py because the upstream version eagerly imports
    # templates with deps we don't ship (langgraph, smolagents, etc.).
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    'random': Random,
    'myagent': MyAgent,
}
""")

    # Point the framework at the gateway sidecar.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Run it. The gateway records every action and emits submission.parquet.
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Save-and-run-all (commit) mode: emit a dummy submission so the
    # commit succeeds. The real submission.parquet is produced by the
    # gateway during competition rerun.
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()
